> **Sentetik veri.** Bu defter `python -m sentetik.uret` ile üretilen uydurma veriyle çalışır. Sayılar gerçek bir şirkete ait değildir ve gerçek projenin sonuçlarını temsil etmez; amaç yöntemi uçtan uca göstermektir. *(Synthetic data: numbers are not real results.)*

# Sinyal — 05 · Aksiyon matrisi, kâr eğrisi ve kontrol grubu tasarımı

**Girdi:** `data/04_skorlar_kalibre_test.parquet` (test kesimi 2025-10-01, LightGBM B, kalibre olasılık M1+M2).

**Dürüstlük notu (sunuma da girecek):** Model *kimin alacağını* tahmin eder, *kime dokunmanın satışı artıracağını* değil. Bu yüzden:
- Aksiyon matrisindeki öneriler **hipotezdir**.
- Kâr eğrisi, temasın etkisi (artımlılık) için açıkça yazılmış **varsayım senaryolarına** dayanır.
- Gerçek etkiyi ancak önerilen **kontrol grubu** ölçer.

**Maskeleme (KVKK):** Müşteri sayısı ve TL yok. Paylar % olarak, değerler **medyan sepet = 1** birimiyle, kâr **popülasyondaki her 1.000 müşteri başına medyan sepet** cinsinden.

In [ ]:
import sys
from pathlib import Path
KOK = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(KOK))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

from src.veri import tablo_yukle
from src.ozellikler import endeksle

SONUC = KOK / "docs" / "sonuclar"; FIG = KOK / "notebooks" / "figurler"; VERI = KOK / "data"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.float_format", "{:.3f}".format)

k = pd.read_parquet(VERI / "04_skorlar_kalibre_test.parquet")
a = endeksle(tablo_yukle("egitim_a"))
a = a[a["kesim"] == k["kesim"].iloc[0]]
medyan_sepet = a["ort_sepet_9ay"].median()
a = a.assign(deger_9ay=a["ciro_9ay"].fillna(0) / medyan_sepet,          # geçmiş değer, medyan sepet birimi
             sepet=(a["ort_sepet_9ay"] / medyan_sepet).fillna(1.0))     # beklenen sipariş büyüklüğü
d = k.merge(a[["kesim", "musteri_id", "deger_9ay", "sepet"]], on=["kesim", "musteri_id"], how="left", validate="one_to_one")
d["y_ciro_90g_e"] = d["y_ciro_90g"] / medyan_sepet
assert len(d) == len(k)
print("test kesimi:", d["kesim"].iloc[0].date(), "| gerçek alım oranı:", round(d["y_etiket"].mean(), 4),
      "| ort. kalibre olasılık:", round(d["p_kalibre"].mean(), 4))

## 1. Varsayımlar (değiştirilebilir; sunumda senaryo olarak)
- **Marj oranı:** Satış tutarının (KDV hariç) brüt kâra kalan payı. İndirimli satışlar dahil ortalama: %40.
- **Temas maliyeti:** Push/e-posta neredeyse bedava (medyan sepetin %0,05'i). SMS medyan sepetin %0,3'ü.
- **Kupon:** Kuponlu senaryoda listedeki **her alıcı** (zaten alacak olanlar dahil) %10 indirim kullanır.
- **Artımlılık (en belirsiz varsayım):** Temas, kişinin alma olasılığını `Δp = u × p × (1 − p) × 4` kadar artırır. En çok orta olasılıklılar etkilenir: kesin alacak ya da hiç almayacak kişiye dokunmak az şey değiştirir. p = 0,5'te artış u'ya eşittir.
  - Senaryolar: u = 1 / 3 / 6 puan (ucuz kanal), kuponlu kanalda bunların 2 katı.

In [ ]:
V = {
    "marj_orani": 0.40,
    "maliyet_ucuz": 0.0005,     # push / e-posta, medyan sepet birimi
    "maliyet_sms": 0.003,
    "kupon_orani": 0.10,
    "u_senaryo": {"karamsar": 0.01, "orta": 0.03, "iyimser": 0.06},
    "kupon_carpani": 2.0,       # kupon, artımlılığı 2 katına çıkarır (varsayım)
}
V

## 2. Aksiyon matrisi: olasılık × geçmiş değer

In [ ]:
d["olasilik"] = pd.cut(d["p_kalibre"], [-1, 0.10, 0.30, 2], labels=["düşük (<%10)", "orta (%10–30)", "yüksek (≥%30)"])
esik = d.loc[d["deger_9ay"] > 0, "deger_9ay"].quantile([1/3, 2/3]).to_numpy()
d["deger"] = pd.cut(d["deger_9ay"], [-1, esik[0], esik[1], np.inf], labels=["düşük", "orta", "yüksek"])

AKSIYON = {   # HİPOTEZ — kontrol grubuyla sınanacak
    ("yüksek (≥%30)", "yüksek"): "İndirim YOK. Yeni sezon / sadakat iletişimi (ucuz kanal).",
    ("yüksek (≥%30)", "orta"):   "İndirim yok; ucuz kanalla hatırlatma.",
    ("yüksek (≥%30)", "düşük"):  "Sepet büyütme (çapraz satış) önerisi.",
    ("orta (%10–30)", "yüksek"): "ÖNCELİKLİ HEDEF: kişiselleştirilmiş push/e-posta, küçük teşvik test edilir.",
    ("orta (%10–30)", "orta"):   "Ucuz kanal + davranışa göre (sepet/favori) hatırlatma.",
    ("orta (%10–30)", "düşük"):  "Yalnızca ucuz kanal.",
    ("düşük (<%10)", "yüksek"):  "Geri kazanım (win-back): kupon testi, küçük grupla.",
    ("düşük (<%10)", "orta"):    "Düşük maliyetli geri kazanım; reklam kitlesi.",
    ("düşük (<%10)", "düşük"):   "Dokunma (maliyet > beklenen getiri).",
}
AM = (d.groupby(["olasilik", "deger"], observed=True)
        .agg(pay=("y_etiket", "size"), ort_olasilik=("p_kalibre", "mean"), gercek_oran=("y_etiket", "mean"),
             ciro_90g_e=("y_ciro_90g_e", "mean"))
        .reset_index())
AM["pay"] = AM["pay"] / len(d)                                   # popülasyon payı (sayı değil)
AM["aksiyon"] = [AKSIYON[(o, v)] for o, v in zip(AM["olasilik"], AM["deger"])]
AM.to_csv(SONUC / "05_aksiyon_matrisi.csv", index=False)
AM

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, (kol, baslik, fmt) in zip(axs, [("pay", "Popülasyon payı", "{:.0%}"),
                                         ("gercek_oran", "Gerçekleşen 90 günlük alım oranı (test)", "{:.0%}")]):
    t = AM.pivot(index="olasilik", columns="deger", values=kol).iloc[::-1]
    im = ax.imshow(t.to_numpy(dtype=float), cmap="Blues")
    ax.set_xticks(range(3)); ax.set_xticklabels([f"değer: {c}" for c in t.columns])
    ax.set_yticks(range(3)); ax.set_yticklabels([f"olasılık: {r}" for r in t.index])
    for i in range(3):
        for j in range(3):
            v = t.iloc[i, j]
            ax.text(j, i, fmt.format(v), ha="center", va="center", color="white" if v > t.to_numpy().max() * 0.6 else "black")
    ax.set_title(baslik)
plt.tight_layout(); plt.savefig(FIG / "05_aksiyon_matrisi.png", bbox_inches="tight"); plt.show()

## 3. Kâr eğrisi (varsayım senaryoları)

In [ ]:
p = d["p_kalibre"].to_numpy(); s = d["sepet"].to_numpy(); n = len(d)
pay_adim = np.linspace(0.01, 1, 100)

def kar_egrisi(u, maliyet, kupon=0.0):
    dp = np.clip(u * p * (1 - p) * 4, 0, 1 - p)
    deger = dp * s * V["marj_orani"] - kupon * s * (p + dp) - maliyet   # kişi başı beklenen artımlı kâr
    sira = np.argsort(-deger, kind="mergesort")                         # beklenen artımlı kâra göre sırala
    kum = np.cumsum(deger[sira])
    return np.array([kum[max(1, int(round(q * n))) - 1] for q in pay_adim]) / n * 1000   # 1.000 müşteri başına

KE, ozet = {}, []
for ad, u in V["u_senaryo"].items():
    KE[("ucuz kanal", ad)] = kar_egrisi(u, V["maliyet_ucuz"])
    KE[("SMS + %10 kupon", ad)] = kar_egrisi(u * V["kupon_carpani"], V["maliyet_sms"], V["kupon_orani"])
for (strateji, ad), egri in KE.items():
    i = int(np.argmax(egri))
    ozet.append({"strateji": strateji, "senaryo": ad, "en_iyi_hedef_payi": pay_adim[i] if egri[i] > 0 else 0.0,
                 "en_yuksek_kar_1000_musteri": max(egri[i], 0.0), "herkese_gonder_kar": egri[-1]})
KO = pd.DataFrame(ozet); KO.to_csv(SONUC / "05_kar_ozeti.csv", index=False)
pd.DataFrame({f"{s}|{a}": e for (s, a), e in KE.items()}, index=pay_adim.round(2)).to_csv(SONUC / "05_kar_egrisi.csv")
KO

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4.2), sharey=False)
for ax, strateji in zip(axs, ["ucuz kanal", "SMS + %10 kupon"]):
    for ad, renk in zip(V["u_senaryo"], ["#9aa0a6", "#1a73e8", "#34a853"]):
        ax.plot(pay_adim * 100, KE[(strateji, ad)], color=renk, label=f"{ad} (u={V['u_senaryo'][ad]:.0%})")
    ax.axhline(0, color="#555", lw=1)
    ax.set_title(f"Kâr eğrisi — {strateji}"); ax.set_xlabel("Hedeflenen müşteri payı (%, beklenen artımlı kâra göre)")
    ax.legend()
axs[0].set_ylabel("Beklenen artımlı kâr\n(1.000 müşteri başına, medyan sepet birimi)")
plt.tight_layout(); plt.savefig(FIG / "05_kar_egrisi.png", bbox_inches="tight"); plt.show()

## 4. Kontrol grubu tasarımı
Her aksiyon hücresinde rastgele bir **kontrol grubu** ayrılır. Bu gruba hiçbir şey gönderilmez. 90 gün sonra, temas edilen grubun alım oranı kontrol grubunun alım oranıyla karşılaştırılır; aradaki fark temasın **gerçek etkisi**dir. Aşağıdaki tablo, belirli bir etkiyi (%95 güven, %80 güç) yakalamak için gereken grup büyüklüğünü **hücre büyüklüğünün yüzdesi** olarak verir.

In [ ]:
def n_grup(p0, fark, alfa=0.05, guc=0.80):
    z = norm.ppf(1 - alfa / 2) + norm.ppf(guc)
    p1 = p0 + fark
    return int(np.ceil(z ** 2 * (p0 * (1 - p0) + p1 * (1 - p1)) / fark ** 2))

KG = []
for _, r in AM.iterrows():
    hucre = r["pay"] * n
    for fark in [0.01, 0.02]:
        g = n_grup(r["gercek_oran"], fark)
        KG.append({"olasilik": r["olasilik"], "deger": r["deger"], "taban_oran": r["gercek_oran"],
                   "yakalanacak_fark_puan": fark * 100,
                   "kontrol_icin_hucre_payi": min(g / hucre, 1.0)})   # kontrol grubu = hücrenin bu kadarı
KG = pd.DataFrame(KG); KG.to_csv(SONUC / "05_kontrol_grubu.csv", index=False)
KG.pivot_table(index=["olasilik", "deger"], columns="yakalanacak_fark_puan", values="kontrol_icin_hucre_payi", observed=True)

## 5. Okuma notları (doldurulacak)
- Aksiyon matrisi: …
- Kâr eğrisi: … (hangi senaryoda kaçta kaçına gönderilmeli; kupon ne zaman kâr ettirir?)
- Kontrol grubu: … (hangi hücrelerde %10 yeterli, hangilerinde daha büyük grup gerekir?)